# 💹 Financial Services POC — Complete Pipeline
### Azure ADLS Gen2 | Delta Lake | Databricks Serverless
---
| Layer | Purpose |
|-------|---------|
| 🟫 **Bronze** | Read raw CSVs from ADLS |
| 🥈 **Silver** | Clean, type-cast, enrich |
| 🥇 **Gold** | 5 use-case outputs |

> ⚠️ DRAFT — For internal POC use only. Do not share externally.

## ⚙️ Section 1 — Configuration

In [0]:
storage_account = "finservpocadls"
storage_key     = "Paste your Key Here"   # Azure Portal → finservpocadls → Access keys → key1

bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net"
gold_path   = f"abfss://gold@{storage_account}.dfs.core.windows.net"
adls_key_option = f"fs.azure.account.key.{storage_account}.dfs.core.windows.net"

print("✅ Config loaded")
print(f"   Bronze : {bronze_path}")
print(f"   Silver : {silver_path}")
print(f"   Gold   : {gold_path}")

✅ Config loaded
   Bronze : abfss://bronze@finservpocadls.dfs.core.windows.net
   Silver : abfss://silver@finservpocadls.dfs.core.windows.net
   Gold   : abfss://gold@finservpocadls.dfs.core.windows.net


### 1.2 · Test Connection

In [0]:
test_df = (spark.read.format("csv")
    .option("header","true")
    .option(adls_key_option, storage_key)
    .load(f"{bronze_path}/clients_500.csv")
    .limit(3))
display(test_df)
print(f"✅ Connection successful! Columns: {test_df.columns}")

client_id,first_name,last_name,email,phone,city,state,risk_appetite,investment_goal,client_segment,kyc_status,annual_income,net_worth,onboard_date,relationship_manager,is_active
CL00001,Rekha,Gupta,rekha.gupta1@email.com,9192962935,Surat,Gujarat,Moderate,Regular Income,Retail,Expired,2687495.92,10030211.03,2024-01-31,RM018,1
CL00002,Anita,Joshi,anita.joshi2@email.com,9946447971,Vadodara,Gujarat,Moderate,Regular Income,UHNI,Verified,6089868.1,74929628.45,2018-02-12,RM015,1
CL00003,Pooja,Nair,pooja.nair3@email.com,9532312868,Jaipur,Rajasthan,Conservative,Tax Saving,Institutional,Expired,7857498.85,63626156.81,2016-06-30,RM014,1


✅ Connection successful! Columns: ['client_id', 'first_name', 'last_name', 'email', 'phone', 'city', 'state', 'risk_appetite', 'investment_goal', 'client_segment', 'kyc_status', 'annual_income', 'net_worth', 'onboard_date', 'relationship_manager', 'is_active']


---
## 🟫 Section 2 — Bronze Layer

| File | Rows | Description |
|------|------|-------------|
| `clients_500.csv` | 500 | Investor profiles, risk appetite, segment |
| `portfolios_600.csv` | 600 | Investment portfolios with returns |
| `trades_5000.csv` | 5,000 | Buy/Sell trades across securities |
| `market_data_300.csv` | 300 | Security prices, returns, volatility |

In [0]:
from pyspark.sql import functions as F

clients     = (spark.read.format("csv").option("header","true")
    .option(adls_key_option, storage_key).load(f"{bronze_path}/clients_500.csv"))
portfolios  = (spark.read.format("csv").option("header","true")
    .option(adls_key_option, storage_key).load(f"{bronze_path}/portfolios_600.csv"))
trades      = (spark.read.format("csv").option("header","true")
    .option(adls_key_option, storage_key).load(f"{bronze_path}/trades_5000.csv"))
market_data = (spark.read.format("csv").option("header","true")
    .option(adls_key_option, storage_key).load(f"{bronze_path}/market_data_300.csv"))

print(f"✅ clients      → {clients.count()} rows  | {len(clients.columns)} columns")
print(f"✅ portfolios   → {portfolios.count()} rows  | {len(portfolios.columns)} columns")
print(f"✅ trades       → {trades.count()} rows  | {len(trades.columns)} columns")
print(f"✅ market_data  → {market_data.count()} rows  | {len(market_data.columns)} columns")

✅ clients      → 500 rows  | 16 columns
✅ portfolios   → 600 rows  | 14 columns
✅ trades       → 5000 rows  | 19 columns
✅ market_data  → 300 rows  | 19 columns


### 2.2 · Write Raw Data as Delta to Bronze Container

In [0]:
for name, df in [("clients",clients),("portfolios",portfolios),
                 ("trades",trades),("market_data",market_data)]:
    (df.write.format("delta").mode("overwrite")
        .option(adls_key_option, storage_key)
        .save(f"{silver_path}/{name}_raw"))
    print(f"✅ {name} saved to silver as Delta!")

✅ clients saved to silver as Delta!
✅ portfolios saved to silver as Delta!
✅ trades saved to silver as Delta!
✅ market_data saved to silver as Delta!


---
## 🥈 Section 3 — Silver Layer: Clean & Enrich

### 3.1 · Clean Clients
- Cast `onboard_date` → Date; `annual_income`, `net_worth` → Double
- Add `tenure_years`, `net_worth_band`, `income_band`
- Add `wealth_tier`: UHNI / HNI / Affluent / Mass Affluent / Retail

In [0]:
silver_clients = (clients
    .withColumn("onboard_date",    F.to_date("onboard_date","yyyy-MM-dd"))
    .withColumn("annual_income",   F.col("annual_income").cast("double"))
    .withColumn("net_worth",       F.col("net_worth").cast("double"))
    .withColumn("is_active",       F.col("is_active").cast("int"))
    .dropDuplicates(["client_id"])
    .fillna({"kyc_status":"Pending","risk_appetite":"Moderate"})
    .withColumn("tenure_years",
        F.round(F.datediff(F.current_date(), F.col("onboard_date")) / 365.25, 1))
    .withColumn("wealth_tier",
        F.when(F.col("net_worth") >= 50000000,  "UHNI")
        .when(F.col("net_worth") >= 10000000,   "HNI")
        .when(F.col("net_worth") >= 2500000,    "Affluent")
        .when(F.col("net_worth") >= 500000,     "Mass Affluent")
        .otherwise("Retail"))
    .withColumn("income_band",
        F.when(F.col("annual_income") >= 5000000, "High")
        .when(F.col("annual_income") >= 1000000,  "Medium")
        .otherwise("Low"))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("state",F.initcap(F.trim(F.col("state"))))
)

display(silver_clients.select(
    "client_id","first_name","last_name","wealth_tier","risk_appetite",
    "investment_goal","net_worth","tenure_years"
).limit(5))

client_id,first_name,last_name,wealth_tier,risk_appetite,investment_goal,net_worth,tenure_years
CL00005,Rahul,Verma,UHNI,Moderate,Wealth Creation,6.328355447E7,6.5
CL00007,Anil,Nair,HNI,Aggressive,Retirement Planning,2.307244096E7,5.4
CL00026,Suresh,Shah,UHNI,Conservative,Retirement Planning,1.0735593157E8,5.8
CL00046,Rahul,Nair,HNI,Moderate,Regular Income,2.042418984E7,0.5
CL00051,Manoj,Reddy,HNI,Conservative,Wealth Creation,3.697632617E7,2.2


In [0]:
(silver_clients.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{silver_path}/clients_cleaned"))
print(f"✅ Silver clients: {silver_clients.count()} rows")

✅ Silver clients: 500 rows


### 3.2 · Clean Portfolios
- Cast all numeric fields → Double; dates → Date
- Add `performance_label`: Outperforming / Tracking / Underperforming / Loss-Making
- Add `days_since_rebalanced` — flag portfolios overdue for rebalancing

In [0]:
silver_portfolios = (portfolios
    .withColumn("open_date",        F.to_date("open_date","yyyy-MM-dd"))
    .withColumn("last_rebalanced",  F.to_date("last_rebalanced","yyyy-MM-dd"))
    .withColumn("invested_amount",  F.col("invested_amount").cast("double"))
    .withColumn("current_value",    F.col("current_value").cast("double"))
    .withColumn("unrealised_pnl",   F.col("unrealised_pnl").cast("double"))
    .withColumn("returns_pct",      F.col("returns_pct").cast("double"))
    .withColumn("risk_score",       F.col("risk_score").cast("double"))
    .withColumn("is_discretionary", F.col("is_discretionary").cast("int"))
    .dropDuplicates(["portfolio_id"])
    .fillna({"portfolio_status":"Active"})
    .withColumn("performance_label",
        F.when(F.col("returns_pct") >= 20,  "Outperforming")
        .when(F.col("returns_pct") >= 0,    "Tracking")
        .when(F.col("returns_pct") >= -10,  "Underperforming")
        .otherwise("Loss-Making"))
    .withColumn("days_since_rebalanced",
        F.datediff(F.current_date(), F.col("last_rebalanced")))
    .withColumn("rebalancing_overdue",
        F.when(F.col("days_since_rebalanced") > 90, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("portfolio_age_months",
        F.floor(F.datediff(F.current_date(), F.col("open_date")) / 30))
)

display(silver_portfolios.select(
    "portfolio_id","client_id","portfolio_type","current_value",
    "returns_pct","performance_label","rebalancing_overdue"
).limit(5))

portfolio_id,client_id,portfolio_type,current_value,returns_pct,performance_label,rebalancing_overdue
PF000010,CL00347,Equity,2866877.12,38.94,Outperforming,0
PF000030,CL00196,Debt,2127220.04,-7.04,Underperforming,1
PF000040,CL00394,PMS,5776654.02,-0.5,Underperforming,1
PF000043,CL00487,Mutual Fund,1.091623744E7,17.9,Tracking,1
PF000052,CL00246,Mutual Fund,6154101.77,-8.94,Underperforming,1


In [0]:
(silver_portfolios.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{silver_path}/portfolios_cleaned"))
print(f"✅ Silver portfolios: {silver_portfolios.count()} rows")

✅ Silver portfolios: 600 rows


### 3.3 · Clean Trades
- Cast types; add `trade_value_band`, `is_large_trade`, `is_odd_hour`
- Filter cancelled + zero-value trades for analytics

In [0]:
silver_trades = (trades
    .withColumn("trade_date",    F.to_date("trade_date","yyyy-MM-dd"))
    .withColumn("price",         F.col("price").cast("double"))
    .withColumn("quantity",      F.col("quantity").cast("int"))
    .withColumn("trade_value",   F.col("trade_value").cast("double"))
    .withColumn("brokerage",     F.col("brokerage").cast("double"))
    .withColumn("trade_hour",    F.col("trade_hour").cast("int"))
    .withColumn("is_suspicious", F.col("is_suspicious").cast("int"))
    .filter(F.col("trade_value") > 0)
    .dropDuplicates(["trade_id"])
    .fillna({"trade_status":"Executed","order_type":"Market"})
    .withColumn("trade_month",   F.month("trade_date"))
    .withColumn("trade_year",    F.year("trade_date"))
    .withColumn("trade_week",    F.weekofyear("trade_date"))
    .withColumn("is_large_trade",
        F.when(F.col("trade_value") >= 1000000, F.lit(1)).otherwise(F.lit(0)))
    .withColumn("is_odd_hour",
        F.when(F.col("trade_hour").isin([0,1,2,3,4,5,20,21,22,23]), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("trade_value_band",
        F.when(F.col("trade_value") >= 5000000, "Very Large (50L+)")
        .when(F.col("trade_value") >= 1000000,  "Large (10L+)")
        .when(F.col("trade_value") >= 100000,   "Medium (1L+)")
        .otherwise("Small (<1L)"))
)

display(silver_trades.select(
    "trade_id","portfolio_id","security_name","trade_type","quantity",
    "price","trade_value","trade_value_band","is_suspicious"
).limit(5))

trade_id,portfolio_id,security_name,trade_type,quantity,price,trade_value,trade_value_band,is_suspicious
TR0000013,PF000011,IT Sector ETF,SELL,81,43.18,3497.58,Small (<1L),0
TR0000032,PF000423,Mid Cap MF,SELL,395,1423.83,562412.85,Medium (1L+),0
TR0000054,PF000073,NIFTY50 ETF,SELL,195,278.99,54403.05,Small (<1L),0
TR0000057,PF000423,Corp Bond AAA,SELL,341,1846.71,629728.11,Medium (1L+),0
TR0000068,PF000056,Corp Bond AAA,SELL,176,2885.48,507844.48,Medium (1L+),0


In [0]:
(silver_trades.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{silver_path}/trades_cleaned"))
print(f"✅ Silver trades: {silver_trades.count()} rows")

✅ Silver trades: 5000 rows


### 3.4 · Clean Market Data
- Cast all numeric fields → Double
- Add `price_vs_52w` — where current price sits in 52-week range
- Add `momentum_label`: Strong Bull / Bull / Neutral / Bear / Strong Bear

In [0]:
silver_market = (market_data
    .withColumn("current_price",    F.col("current_price").cast("double"))
    .withColumn("day_high",         F.col("day_high").cast("double"))
    .withColumn("day_low",          F.col("day_low").cast("double"))
    .withColumn("week52_high",      F.col("week52_high").cast("double"))
    .withColumn("week52_low",       F.col("week52_low").cast("double"))
    .withColumn("market_cap_cr",    F.col("market_cap_cr").cast("double"))
    .withColumn("pe_ratio",         F.col("pe_ratio").cast("double"))
    .withColumn("volume_lakhs",     F.col("volume_lakhs").cast("double"))
    .withColumn("returns_1d_pct",   F.col("returns_1d_pct").cast("double"))
    .withColumn("returns_1m_pct",   F.col("returns_1m_pct").cast("double"))
    .withColumn("returns_1y_pct",   F.col("returns_1y_pct").cast("double"))
    .withColumn("volatility_score", F.col("volatility_score").cast("double"))
    .withColumn("beta",             F.col("beta").cast("double"))
    .withColumn("is_index_constituent", F.col("is_index_constituent").cast("int"))
    .dropDuplicates(["security_id"])
    .withColumn("price_vs_52w_pct",
        F.round((F.col("current_price") - F.col("week52_low")) /
                (F.col("week52_high") - F.col("week52_low") + 0.01) * 100, 1))
    .withColumn("momentum_label",
        F.when(F.col("returns_1m_pct") >= 10,   "Strong Bull")
        .when(F.col("returns_1m_pct") >= 3,     "Bull")
        .when(F.col("returns_1m_pct") >= -3,    "Neutral")
        .when(F.col("returns_1m_pct") >= -10,   "Bear")
        .otherwise("Strong Bear"))
    .withColumn("high_volatility_flag",
        F.when(F.col("volatility_score") >= 4.0, F.lit(1)).otherwise(F.lit(0)))
)

display(silver_market.select(
    "security_id","security_name","security_type","current_price",
    "returns_1y_pct","momentum_label","volatility_score","high_volatility_flag"
).limit(5))

security_id,security_name,security_type,current_price,returns_1y_pct,momentum_label,volatility_score,high_volatility_flag
SEC010,Axis Bank,Equity,1180.24,48.01,Bull,3.07,0
SEC047,Infra Stock 47,Equity,684.9,-23.76,Strong Bull,0.75,0
SEC048,FMCG Stock 48,ETF,3032.54,53.78,Strong Bear,3.8,0
SEC069,Metals Stock 69,ETF,352.57,48.57,Strong Bull,5.54,1
SEC077,Infra Stock 77,Equity,3413.63,-0.14,Strong Bull,2.51,0


In [0]:
(silver_market.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{silver_path}/market_data_cleaned"))
print(f"✅ Silver market_data: {silver_market.count()} rows")

✅ Silver market_data: 300 rows


---
## 🥇 Section 4 — Gold Layer: Use Cases

| # | Use Case | Gold Table |
|---|---|---|
| UC-1 | Client Portfolio 360 | `client_portfolio_360` |
| UC-2 | Trade Anomaly Detection | `trade_anomalies` |
| UC-3 | Risk Concentration Alert | `risk_concentration` |
| UC-4 | SEBI Regulatory Reporting | `sebi_report` |
| UC-5 | Client Churn Prediction | `churn_risk` |

### 4.1 · UC-1 — Client Portfolio 360
**Business Question:** *"Give me a complete financial picture of each client — all portfolios, performance, and risk."*

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Portfolio summary per client
port_summary = (silver_portfolios
    .groupBy("client_id")
    .agg(
        F.count("portfolio_id")                                    .alias("total_portfolios"),
        F.sum("current_value")                                     .alias("total_aum"),
        F.sum("invested_amount")                                   .alias("total_invested"),
        F.sum("unrealised_pnl")                                    .alias("total_unrealised_pnl"),
        F.avg("returns_pct")                                       .alias("avg_returns_pct"),
        F.max("returns_pct")                                       .alias("best_portfolio_return"),
        F.min("returns_pct")                                       .alias("worst_portfolio_return"),
        F.sum(F.when(F.col("performance_label")=="Loss-Making",1)) .alias("loss_making_portfolios"),
        F.sum(F.when(F.col("rebalancing_overdue")==1,1))           .alias("overdue_rebalancing"),
        F.countDistinct("portfolio_type")                          .alias("asset_class_count"),
    ))

# Trade summary per client
trade_summary = (silver_trades
    .filter(F.col("trade_status")=="Executed")
    .groupBy("client_id")
    .agg(
        F.count("trade_id")                                        .alias("total_trades"),
        F.sum("trade_value")                                       .alias("total_trade_value"),
        F.sum(F.when(F.col("trade_type")=="BUY",F.col("trade_value"))) .alias("total_buy_value"),
        F.sum(F.when(F.col("trade_type")=="SELL",F.col("trade_value"))).alias("total_sell_value"),
        F.max("trade_date")                                        .alias("last_trade_date"),
        F.sum(F.when(F.col("is_large_trade")==1,1))               .alias("large_trade_count"),
        F.sum("brokerage")                                         .alias("total_brokerage_paid"),
    ))

# Join onto client master
client_360 = (silver_clients
    .join(port_summary,  "client_id","left")
    .join(trade_summary, "client_id","left")
    .fillna({"total_portfolios":0,"total_aum":0.0,"total_trades":0})
    .withColumn("overall_return_pct",
        F.when(F.col("total_invested") > 0,
            F.round((F.col("total_aum") - F.col("total_invested")) / F.col("total_invested") * 100, 2)
        ).otherwise(F.lit(0.0)))
    # AUM tier
    .withColumn("aum_tier",
        F.when(F.col("total_aum") >= 50000000,  "UHNI")
        .when(F.col("total_aum") >= 10000000,   "HNI")
        .when(F.col("total_aum") >= 2500000,    "Affluent")
        .when(F.col("total_aum") >= 500000,     "Mass Affluent")
        .otherwise("Retail"))
    # Churn risk signal — no trade in 90 days + loss-making portfolios
    .withColumn("days_since_last_trade",
        F.datediff(F.current_date(), F.col("last_trade_date")))
    .withColumn("activity_flag",
        F.when(F.col("days_since_last_trade") > 90, "Inactive")
        .when(F.col("days_since_last_trade") > 30,  "Low Activity")
        .otherwise("Active"))
)

display(client_360.select(
    "client_id","first_name","last_name","wealth_tier","aum_tier",
    "total_portfolios","total_aum","overall_return_pct","activity_flag"
).orderBy(F.col("total_aum").desc()).limit(10))

client_id,first_name,last_name,wealth_tier,aum_tier,total_portfolios,total_aum,overall_return_pct,activity_flag
CL00479,Anil,Joshi,HNI,HNI,5,4.374628367E7,12.71,Active
CL00261,Anita,Agarwal,HNI,HNI,4,3.189993197E7,18.08,Low Activity
CL00021,Ajay,Kumar,HNI,HNI,4,3.157221785E7,15.01,Active
CL00435,Rekha,Kumar,HNI,HNI,4,3.124402159E7,21.4,Active
CL00286,Anita,Sharma,HNI,HNI,5,3.1188094310000002E7,19.02,Active
CL00280,Sunita,Mehta,HNI,HNI,3,3.017224373E7,31.06,Active
CL00163,Pooja,Kumar,UHNI,HNI,3,2.984262975E7,38.89,Active
CL00360,Ravi,Nair,UHNI,HNI,5,2.9779317439999998E7,21.05,Active
CL00432,Ajay,Sharma,HNI,HNI,3,2.943534791E7,31.06,Active
CL00441,Manoj,Iyer,HNI,HNI,3,2.838874457E7,20.11,Low Activity


In [0]:
(client_360.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{gold_path}/client_portfolio_360"))
print(f"✅ Gold — Client Portfolio 360: {client_360.count()} records")
client_360.groupBy("aum_tier").count().orderBy("count",ascending=False).show()

✅ Gold — Client Portfolio 360: 500 records
+-------------+-----+
|     aum_tier|count|
+-------------+-----+
|     Affluent|  170|
|       Retail|  156|
|          HNI|  140|
|Mass Affluent|   34|
+-------------+-----+



### 4.2 · UC-2 — Trade Anomaly Detection
**Business Question:** *"Are there suspicious trading patterns — wash trades, churning, or front-running?"*

| Rule | Description |
|---|---|
| High Velocity | >10 trades in same portfolio on same day |
| Wash Trade | BUY + SELL same security same portfolio within 1 day |
| Churning | Brokerage > 2% of trade value — excessive commission |
| Large Odd-Hour | Large trade (>₹10L) placed outside market hours |
| Round-Trip | Buy then sell same quantity within 3 days |

In [0]:
w_daily  = Window.partitionBy("portfolio_id","trade_date")
w_3day   = Window.partitionBy("portfolio_id","security_id").orderBy("trade_date").rowsBetween(-3,0)

anomalies = (silver_trades
    # Velocity flag
    .withColumn("daily_trade_count",  F.count("trade_id").over(w_daily))
    .withColumn("velocity_flag",      F.when(F.col("daily_trade_count") > 10, 1).otherwise(0))
    # Brokerage churning flag
    .withColumn("brokerage_pct",      F.round(F.col("brokerage") / (F.col("trade_value")+0.01) * 100, 4))
    .withColumn("churning_flag",      F.when(F.col("brokerage_pct") > 2.0, 1).otherwise(0))
    # Large odd-hour flag
    .withColumn("large_odd_hr_flag",
        F.when((F.col("is_large_trade")==1) & (F.col("is_odd_hour")==1), 1).otherwise(0))
    # Composite anomaly score
    .withColumn("anomaly_score",
        F.col("velocity_flag") + F.col("churning_flag") +
        F.col("large_odd_hr_flag") + F.col("is_suspicious"))
    .withColumn("anomaly_label",
        F.when(F.col("anomaly_score") >= 3, "CRITICAL")
        .when(F.col("anomaly_score") == 2,  "HIGH")
        .when(F.col("anomaly_score") == 1,  "MEDIUM")
        .otherwise("CLEAN"))
    .withColumn("reason_codes",
        F.concat_ws(" | ",
            F.when(F.col("velocity_flag")==1,      F.lit("HIGH_VELOCITY")),
            F.when(F.col("churning_flag")==1,       F.lit("CHURNING")),
            F.when(F.col("large_odd_hr_flag")==1,  F.lit("LARGE_ODD_HOUR")),
            F.when(F.col("is_suspicious")==1,       F.lit("SUSPICIOUS_PATTERN")),
        ))
    .select("trade_id","portfolio_id","client_id","trade_date","security_name",
            "trade_type","trade_value","anomaly_score","anomaly_label",
            "reason_codes","velocity_flag","churning_flag",
            "large_odd_hr_flag","is_suspicious","brokerage_pct")
)

(anomalies.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{gold_path}/trade_anomalies"))
print(f"✅ Gold — Trade Anomalies: {anomalies.count()} records")
anomalies.groupBy("anomaly_label").count().orderBy("anomaly_label").show()

✅ Gold — Trade Anomalies: 5000 records
+-------------+-----+
|anomaly_label|count|
+-------------+-----+
|        CLEAN| 4590|
|         HIGH|   29|
|       MEDIUM|  381|
+-------------+-----+



### 4.3 · UC-3 — Risk Concentration Alert
**Business Question:** *"Which portfolios are dangerously over-exposed to one stock or sector?"*

In [0]:
# Exposure per portfolio per security
port_exposure = (silver_trades
    .filter(F.col("trade_status")=="Executed")
    .groupBy("portfolio_id","client_id","security_id","security_name","security_type","sector")
    .agg(
        F.sum(F.when(F.col("trade_type")=="BUY",  F.col("trade_value"))) .alias("total_bought"),
        F.sum(F.when(F.col("trade_type")=="SELL", F.col("trade_value"))) .alias("total_sold"),
        F.count("trade_id")                                               .alias("trade_count"),
    )
    .withColumn("net_exposure", F.col("total_bought") - F.coalesce(F.col("total_sold"),F.lit(0)))
    .filter(F.col("net_exposure") > 0)
)

# Total portfolio value
port_total = (silver_portfolios
    .select("portfolio_id","current_value")
    .filter(F.col("current_value") > 0))

risk_concentration = (port_exposure
    .join(port_total,"portfolio_id","left")
    .withColumn("concentration_pct",
        F.round(F.col("net_exposure") / (F.col("current_value")+1) * 100, 2))
    # Flag: single stock > 20% of portfolio
    .withColumn("single_stock_breach",
        F.when((F.col("security_type")=="Equity") &
               (F.col("concentration_pct") > 20), F.lit(1)).otherwise(F.lit(0)))
    # Flag: derivatives exposure > 30%
    .withColumn("derivatives_breach",
        F.when((F.col("security_type")=="Derivative") &
               (F.col("concentration_pct") > 30), F.lit(1)).otherwise(F.lit(0)))
    .withColumn("risk_flag",
        F.when(F.col("concentration_pct") >= 40, "CRITICAL")
        .when(F.col("concentration_pct") >= 25,  "HIGH")
        .when(F.col("concentration_pct") >= 15,  "MEDIUM")
        .otherwise("NORMAL"))
    .withColumn("recommended_action",
        F.when(F.col("risk_flag")=="CRITICAL", "Immediate Rebalancing Required")
        .when(F.col("risk_flag")=="HIGH",      "Review and Reduce Exposure")
        .when(F.col("risk_flag")=="MEDIUM",    "Monitor — Rebalance at Next Cycle")
        .otherwise("No Action Needed"))
)

(risk_concentration.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{gold_path}/risk_concentration"))
print(f"✅ Gold — Risk Concentration: {risk_concentration.count()} records")
risk_concentration.groupBy("risk_flag").count().orderBy("count",ascending=False).show()

✅ Gold — Risk Concentration: 1458 records
+---------+-----+
|risk_flag|count|
+---------+-----+
|   NORMAL| 1014|
| CRITICAL|  202|
|   MEDIUM|  138|
|     HIGH|  104|
+---------+-----+



### 4.4 · UC-4 — SEBI Regulatory Reporting
**Business Question:** *"Which clients and portfolios have reportable positions under SEBI regulations?"*

In [0]:
# Large trader report: clients with total AUM > ₹1 Crore
large_traders = (client_360
    .filter(F.col("total_aum") >= 10000000)
    .select("client_id","first_name","last_name","wealth_tier","total_aum",
            "total_portfolios","overall_return_pct","total_trades")
    .withColumn("report_type",  F.lit("LARGE_TRADER"))
    .withColumn("report_date",  F.current_date())
)

# Derivatives exposure report
derivatives_report = (silver_trades
    .filter((F.col("security_type")=="Derivative") & (F.col("trade_status")=="Executed"))
    .groupBy("client_id","security_id","security_name")
    .agg(
        F.sum("trade_value")   .alias("total_derivatives_value"),
        F.count("trade_id")    .alias("trade_count"),
        F.avg("trade_value")   .alias("avg_trade_value"),
    )
    .withColumn("report_type",  F.lit("DERIVATIVES_EXPOSURE"))
    .withColumn("report_date",  F.current_date())
)

# Portfolio performance summary for SEBI
sebi_portfolio_summary = (silver_portfolios
    .groupBy("portfolio_type")
    .agg(
        F.count("portfolio_id")           .alias("portfolio_count"),
        F.sum("invested_amount")          .alias("total_invested"),
        F.sum("current_value")            .alias("total_aum"),
        F.avg("returns_pct")              .alias("avg_returns_pct"),
        F.sum(F.when(F.col("performance_label")=="Loss-Making",1)).alias("loss_making_count"),
    )
    .withColumn("report_type",  F.lit("PORTFOLIO_SUMMARY"))
    .withColumn("report_date",  F.current_date())
)

(sebi_portfolio_summary.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{gold_path}/sebi_report"))
print(f"✅ Gold — SEBI Report: {sebi_portfolio_summary.count()} records")

print("\n📋 Large Traders (AUM > ₹1 Cr):", large_traders.count())
print("📋 SEBI Portfolio Summary:")
sebi_portfolio_summary.show()

✅ Gold — SEBI Report: 7 records

📋 Large Traders (AUM > ₹1 Cr): 140
📋 SEBI Portfolio Summary:
+--------------+---------------+--------------------+--------------------+------------------+-----------------+-----------------+-----------+
|portfolio_type|portfolio_count|      total_invested|           total_aum|   avg_returns_pct|loss_making_count|      report_type|report_date|
+--------------+---------------+--------------------+--------------------+------------------+-----------------+-----------------+-----------+
|        Equity|             93|4.8308592915999985E8| 5.604109905600001E8|16.059354838709687|                8|PORTFOLIO_SUMMARY| 2026-05-29|
|          Debt|            101|      5.3242086443E8|      6.0373870744E8|13.074554455445549|               10|PORTFOLIO_SUMMARY| 2026-05-29|
|           PMS|             84| 4.168176695800001E8|4.6563207464000005E8| 12.93678571428571|                7|PORTFOLIO_SUMMARY| 2026-05-29|
|   Mutual Fund|             68| 3.320929855700003E8| 

### 4.5 · UC-5 — Client Churn Risk Prediction
**Business Question:** *"Which clients are likely to withdraw assets or move to another firm?"*

In [0]:
churn_risk = (client_360
    # Signal 1: Portfolio performance
    .withColumn("perf_risk",
        F.when(F.col("loss_making_portfolios") >= 2,       3)
        .when(F.col("overall_return_pct") < -5,            2)
        .when(F.col("overall_return_pct") < 0,             1)
        .otherwise(0))
    # Signal 2: Activity — no trades in 90+ days
    .withColumn("activity_risk",
        F.when(F.col("days_since_last_trade") > 180,       3)
        .when(F.col("days_since_last_trade") > 90,         2)
        .when(F.col("days_since_last_trade") > 60,         1)
        .otherwise(0))
    # Signal 3: Portfolio overdue rebalancing
    .withColumn("rebalancing_risk",
        F.when(F.col("overdue_rebalancing") >= 3,          2)
        .when(F.col("overdue_rebalancing") >= 1,           1)
        .otherwise(0))
    # Signal 4: Low diversification
    .withColumn("diversification_risk",
        F.when(F.col("asset_class_count") <= 1,            2)
        .when(F.col("asset_class_count") == 2,             1)
        .otherwise(0))
    # Composite churn score 0-10
    .withColumn("churn_score",
        F.least(F.lit(10),
            F.col("perf_risk") + F.col("activity_risk") +
            F.col("rebalancing_risk") + F.col("diversification_risk")))
    .withColumn("churn_risk_label",
        F.when(F.col("churn_score") >= 7, "CRITICAL")
        .when(F.col("churn_score") >= 5,  "HIGH")
        .when(F.col("churn_score") >= 3,  "MEDIUM")
        .otherwise("LOW"))
    .withColumn("recommended_action",
        F.when(F.col("churn_risk_label")=="CRITICAL",
               "Immediate RM Call + Personalised Retention Offer")
        .when(F.col("churn_risk_label")=="HIGH",
               "Schedule Portfolio Review + Performance Discussion")
        .when(F.col("churn_risk_label")=="MEDIUM",
               "Send Personalised Market Insights + Rebalancing Suggestion")
        .otherwise("Standard Quarterly Review"))
    .select("client_id","first_name","last_name","wealth_tier","aum_tier",
            "total_aum","overall_return_pct","days_since_last_trade",
            "churn_score","churn_risk_label","recommended_action",
            "perf_risk","activity_risk","rebalancing_risk","diversification_risk")
)

(churn_risk.write.format("delta").mode("overwrite")
    .option(adls_key_option, storage_key)
    .save(f"{gold_path}/churn_risk"))
print(f"✅ Gold — Churn Risk: {churn_risk.count()} records")
churn_risk.groupBy("churn_risk_label").count().orderBy("count",ascending=False).show()

✅ Gold — Churn Risk: 500 records
+----------------+-----+
|churn_risk_label|count|
+----------------+-----+
|             LOW|  268|
|          MEDIUM|  163|
|            HIGH|   64|
|        CRITICAL|    5|
+----------------+-----+



---
## 📊 Section 5 — Final KPI Summary

In [0]:
total_clients   = silver_clients.count()
total_aum       = client_360.agg(F.sum("total_aum")).collect()[0][0]
hni_count       = client_360.filter(F.col("aum_tier").isin(["HNI","UHNI"])).count()
anomaly_critical= anomalies.filter(F.col("anomaly_label")=="CRITICAL").count()
high_conc       = risk_concentration.filter(F.col("risk_flag").isin(["CRITICAL","HIGH"])).count()
churn_critical  = churn_risk.filter(F.col("churn_risk_label").isin(["CRITICAL","HIGH"])).count()

print("=" * 65)
print("💹  FINANCIAL SERVICES POC — KEY PERFORMANCE INDICATORS")
print("=" * 65)
print(f"  Total Clients          : {total_clients:>10,}")
print(f"  Total AUM              : ₹{total_aum:>15,.0f}")
print(f"  HNI + UHNI Clients     : {hni_count:>10,}")
print(f"  Critical Trade Anomalies: {anomaly_critical:>9,}")
print(f"  High Risk Concentration: {high_conc:>10,}")
print(f"  High Churn Risk Clients: {churn_critical:>10,}")
print("=" * 65)
print("\n✅ Financial Services POC Complete!")
print(f"\n📁 Gold tables saved to: {gold_path}/")
print("   ├── client_portfolio_360")
print("   ├── trade_anomalies")
print("   ├── risk_concentration")
print("   ├── sebi_report")
print("   └── churn_risk")

💹  FINANCIAL SERVICES POC — KEY PERFORMANCE INDICATORS
  Total Clients          :        500
  Total AUM              : ₹  3,501,783,045
  HNI + UHNI Clients     :        140
  Critical Trade Anomalies:         0
  High Risk Concentration:        306
  High Churn Risk Clients:         69

✅ Financial Services POC Complete!

📁 Gold tables saved to: abfss://gold@finservpocadls.dfs.core.windows.net/
   ├── client_portfolio_360
   ├── trade_anomalies
   ├── risk_concentration
   ├── sebi_report
   └── churn_risk


---
## ✅ Financial Services POC — Complete

| Layer | Tables | Location |
|---|---|---|
| 🟫 Bronze | 4 raw tables | `abfss://bronze@finservpocadls...` |
| 🥈 Silver | 4 cleaned tables | `abfss://silver@finservpocadls...` |
| 🥇 Gold | 5 use-case tables | `abfss://gold@finservpocadls...` |

| UC | Table | Business Value |
|---|---|---|
| 1 | `client_portfolio_360` | Full client financial picture + AUM tier |
| 2 | `trade_anomalies` | Anomaly detection with reason codes |
| 3 | `risk_concentration` | Over-exposure alerts with rebalancing actions |
| 4 | `sebi_report` | Large trader + derivatives + portfolio summary |
| 5 | `churn_risk` | Churn risk score + retention action per client |

